# 00 · Smoke test on a 12-patient subset (run this first)

Runs every CPU-capable stage on 12 seeded patients (each with at least 3 MRIs) using the expert segmentations. Outputs go to `cache_smoke/` and `reports_smoke/` on Drive, so the full run is never touched. Settings: 3 folds, 200 bootstraps, 5 epochs, simple radiomics, and the random3d encoder. A CPU runtime is enough; it takes about 30–60 min. The numbers are meaningless; only crashes and data-layout problems matter.

In [ ]:
# --- setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['BMF_DRIVE'] = '/content/drive/MyDrive/bmets'                       # change if you use another folder
os.environ['BMF_DATA'] = os.environ['BMF_DRIVE'] + '/UCSD-BMETS-Longitudinal'   # Aspera download target (uploaded to Drive)
os.makedirs(os.environ['BMF_DRIVE'], exist_ok=True)
REPO = '/content/claude_projects'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rittik047/claude_projects.git $REPO
%cd /content/claude_projects/bmets-forecast
!pip -q install -e ".[dl,extra]"
CFG = 'configs/colab.yaml'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
N = 12
S = f'--config {CFG} --subset {N}'
for st in ['audit', 'index', 'splits']:
    !bmf {st} {S}

In [ ]:
import pandas as pd
ix = pd.read_csv(os.environ['BMF_DRIVE'] + '/cache_smoke/index.csv')
print(ix.groupby('seq').exists.mean().round(2))   # share of studies with each sequence found
assert ix.query("seq == 't1c'").exists.astype(str).eq('True').all(), 'missing t1c: check BMF_DATA layout'

In [ ]:
import glob, nibabel as nib, numpy as np
f = sorted(glob.glob(os.environ['BMF_DATA'] + '/*/*seg*.nii.gz'))[0]
print(f, np.unique(nib.load(f).get_fdata()))   # expect {0,1,2,3} (+4 resection cavity)

In [ ]:
for st in ['preprocess', 'track', 'labels', 'features']:
    !bmf {st} {S}
!bmf embed {S} --encoder random3d

In [ ]:
for m in ['P0', 'P1', 'P2', 'P3', 'P5']:
    !bmf train {S} --model {m} --task T2a
!bmf train {S} --model P1 --task T3
!bmf train {S} --model O1 --task T4
for st in ['evaluate', 'explain', 'figures', 'report']:
    !bmf {st} {S}

In [ ]:
print(open(os.environ['BMF_DRIVE'] + '/reports_smoke/figures_log.txt').read())   # one line = no figure failed
import json; r = json.load(open(os.environ['BMF_DRIVE'] + '/reports_smoke/results.json'))
print(json.dumps(r['cohort'], indent=1)); print({m: round(v['auroc'], 2) for m, v in r['T2a']['models'].items()})